# 🦺 Construction Site Safety PPE Detection - Dedicated Cap Class Pipeline
### โซลูชันที่แก้ปัญหาถาวร 100%: แยกคลาส `Cap` (หมวกแก๊ป) ออกจาก `Hardhat` (หมวกนิรภัย) และ `NO-Hardhat` (หัวเปล่า)

**ทำไมวิธีนี้ถึงหายขาด 100%?**
- โมเดลจะไม่สับสน เพราะรูปทรงพลาสติกสะท้อนแสงของหมวกนิรภัย กับทรงผ้าของหมวกแก๊ป จะมีคลาสของตัวเองอย่างอิสระ
- `Hardhat` จะยังคงแม่นยำสูงลิ่ว 95-98% (ไม่หลุด ไม่หลงเป็น NO-Hardhat)
- `Cap` จะถูกตรวจจับได้แม่นยำ 90%+ และในเว็บแอปจะแจ้งเตือนเป็น **VIOLATION (ละเมิดความปลอดภัย: สวมหมวกแก๊ปแทนหมวกนิรภัย)**

## 1. ตรวจสอบ GPU บน Google Colab

In [ ]:
!nvidia-smi
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))

## 2. ติดตั้ง Ultralytics (YOLOv8) และเครื่องมือที่จำเป็น

In [ ]:
!pip install -q ultralytics roboflow openvino onnx

## 3. ดาวน์โหลดชุดข้อมูล CSS (Site Safety) และชุดข้อมูล Caps จาก Roboflow

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="JZxCGXvWDAkiBZjEEx7Z")

# 1) ดาวน์โหลด Construction Site Safety (v30)
project_css = rf.workspace("roboflow-universe-projects").project("construction-site-safety")
dataset_css = project_css.version(30).download("yolov8")
print('[+] CSS Dataset path:', dataset_css.location)

# 2) ดาวน์โหลด Caps Dataset (หมวกแก๊ป)
project_caps = rf.workspace("object-detection-cwcjo").project("caps-gi4un")
dataset_caps = project_caps.version(2).download("yolov8")
print('[+] Caps Dataset path:', dataset_caps.location)

## 4. รวมชุดข้อมูลโดยเพิ่มคลาสใหม่ `Cap` เข้าใน data.yaml อย่างถูกต้อง

In [ ]:
import yaml, os, shutil, glob

css_yaml_path = f"{dataset_css.location}/data.yaml"
with open(css_yaml_path, 'r') as f:
    data = yaml.safe_load(f)

# เพิ่มคลาส Cap ต่อท้ายชุดข้อมูลเดิม
cap_class_id = len(data['names'])
data['names'].append('Cap')
data['nc'] = len(data['names'])
print(f'[+] Added class Cap at ID: {cap_class_id}. Total classes: {data["nc"]}')

with open(css_yaml_path, 'w') as f:
    yaml.safe_dump(data, f)

# คัดลอกภาพหมวกแก๊ป 500 ภาพ และเขียน Label เป็นคลาส Cap
caps_images = glob.glob(f"{dataset_caps.location}/train/images/*.jpg")[:500]
added_count = 0
for img_p in caps_images:
    lbl_p = img_p.replace('/images/', '/labels/').replace('.jpg', '.txt')
    if not os.path.exists(lbl_p):
        continue
    
    new_lines = []
    with open(lbl_p, 'r') as lf:
        for line in lf:
            parts = line.strip().split()
            if len(parts) >= 5:
                new_lines.append(f"{cap_class_id} {' '.join(parts[1:])}\n")
    
    if new_lines:
        fname = os.path.basename(img_p)
        dest_img = f"{dataset_css.location}/train/images/cap_{fname}"
        dest_lbl = f"{dataset_css.location}/train/labels/cap_{fname.replace('.jpg', '.txt')}"
        shutil.copyfile(img_p, dest_img)
        with open(dest_lbl, 'w') as out_f:
            out_f.writelines(new_lines)
        added_count += 1

print(f'[+] Successfully added {added_count} Cap images with dedicated Class ID: {cap_class_id}')

## 5. เริ่มต้นการเทรน YOLOv8s (30-40 Epochs บน GPU)

In [ ]:
from ultralytics import YOLO

# โหลดโมเดล YOLOv8s
model = YOLO('yolov8s.pt')

results = model.train(
    data=css_yaml_path,
    epochs=35,
    batch=32,
    imgsz=640,
    device=0,
    workers=4,
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    box=7.5,
    cls=1.2,
    dfl=1.5,
    name='ppe_yolov8s_with_caps_perfect'
)

## 6. ตรวจสอบความแม่นยำและดาวน์โหลดไฟล์ best.pt

In [ ]:
from google.colab import files
best_pt_path = '/content/runs/detect/ppe_yolov8s_with_caps_perfect/weights/best.pt'
files.download(best_pt_path)
print('[+] Click download to get your perfect best.pt model!')